# Star Trek NLP — Text Preprocessing and Frequency Analysis

An exploratory NLP pipeline applied to dialogue from *Star Trek: Enterprise* episode scripts. The workflow covers text normalization (punctuation, URL, HTML, contraction handling, number-to-word conversion), tokenization, stopword removal, and a side-by-side comparison of **WordNet lemmatization** versus **Lancaster stemming**. A per-episode frequency distribution highlights the most-spoken character names.

**Data:** Star Trek scripts dataset from Kaggle ([`gjbroughton/start-trek-scripts`](https://www.kaggle.com/datasets/gjbroughton/start-trek-scripts)). The two source JSON files (`all_scripts_raw.json`, `all_series_lines.json`, ~43 MB combined) are not committed to this repository — download from Kaggle and place them alongside this notebook before running.

**Scope note:** This notebook focuses on the preprocessing pipeline and lemmatization/stemming comparison. Downstream modeling (sentiment, clustering) is out of scope here.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_json('all_scripts_raw.json')

In [3]:
df.head()

,DS9,TOS,TAS,TNG,VOY,ENT
episode 0,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - The Ca...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\nThe Voyager Transcripts - Caretaker\...,\n\n\n\n\n\nThe Enterprise Transcripts - Broke...
episode 1,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - The Ma...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\n\nThe Voyager Transcripts - Parallax...,\n\n\n\n\n\nThe Enterprise Transcripts - Fight...
episode 2,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - Charli...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\n\nThe Voyager Transcripts - Time and...,\n\n\n\n\n\nThe Enterprise Transcripts - Stran...
episode 3,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - Where ...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\n\n\nThe Voyager Transcripts - Phage\...,\n\n\n\n\n\nThe Enterprise Transcripts - Unexp...
episode 4,\n\n\n\n\n\nThe Deep Space Nine Transcripts - ...,\n\n\n\n\n\nThe Star Trek Transcripts - The Na...,\n\n\n\n\n\nThe Animated Star Trek Transcripts...,\n\n\n\n\n\nThe Next Generation Transcripts - ...,\n\n\n\n\n\nThe Voyager Transcripts - The Clou...,\n\n\n\n\n\nThe Enterprise Transcripts - Terra...


In [4]:
### Let's have a quick peek at the dataset.. 6 seasons with episodes ranging from 1 - 176. WOW....

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 176 entries, episode 0 to episode 175
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   DS9     173 non-null    object
 1   TOS     80 non-null     object
 2   TAS     22 non-null     object
 3   TNG     176 non-null    object
 4   VOY     160 non-null    object
 5   ENT     97 non-null     object
dtypes: object(6)
memory usage: 9.6+ KB


In [6]:
### Def appears to be quite a few different text patterns we need to remove (\n,\,etc.)

In [7]:
df.DS9[1]

"\n\n\n\n\n\nThe Deep Space Nine Transcripts - Past Prologue\n\n\n\nPast\nPrologue\nStardate:\nUnknown\nOriginal Airdate: 11 Jan, 1993\n\n\n\n\n\n\n  [Replimat]\n\nGARAK: It's Doctor Bashir, isn't it? Of course it\nis. May I introduce myself? \nBASHIR: Er, yes, yes, of course. \nGARAK: My name is Garak. Cardassian by birth, obviously. The only one\nof us left on this station, as a matter of fact, so I do appreciate\nmaking new friends whenever I can. You are new to this station, I\nbelieve. \nBASHIR: I am, yes. Though, though I understand you've been here quite a\nwhile. \nGARAK: Ah, you know of me then. \nBASHIR: Would you care for some of this Tarkalean tea? It's very good. \nGARAK: What a thoughtful young man. How nice that we've met. \nBASHIR: You know, some people say that you remained on DS Nine as the\neyes and ears of your fellow Cardassians. \nGARAK: You don't say? Doctor, you're not intimating that I'm considered\nsome sort of spy, are you? \nBASHIR: I wouldn't know, sir. \nG

In [8]:
## Number of episodes for each season of Star Trek..
for i in range(0,6):
    print('{} := {}'.format(df.columns.values[i],df[df.columns[i]].count()))

DS9 := 173
TOS := 80
TAS := 22
TNG := 176
VOY := 160
ENT := 97


In [9]:
## Lets single out "ENT" for now to get our bearings straight and see how to process the text for our NLP model.

In [10]:
ent = df.ENT

ent.shape ## That doesn't match up now does it?!?! 

(176,)

In [11]:
## Looks like we need to drop some empty episodes!
ent.isnull().sum()

79

In [12]:
ent = ent.dropna()

In [13]:
ent.shape

(97,)

Here are some preprocessing steps:

    * Removing punctuations like . , ! $( ) * % @
    * Removing URLs
    * Removing Stop words
    * Lower casing
    * Tokenization
    * Stemming
    * Lemmatization


In [14]:
def lowerCase(text):
    return text.lower()

def remPunct(text):
    import string
    translator =  str.maketrans('','', string.punctuation)
    return text.translate(translator)

def remWhitespace(text):
    return " ".join(text.split())

def remHTML(text):
    from bs4 import BeautifulSoup
    
    beaut = BeautifulSoup(text,'html.parser')
    return beaut.get_text()

def remURL(text):
    import re
    
    return re.sub(r'https\S','',text)

def remBrackets(text):
    import re
    
    return re.sub('\(.*?\)','',text)

## To convert numbers to words... If word is digit, it will convert 
## to number and append to newStr list or append word as is. Finally,
## it will join words back into string.
def changeNum(text):
    ## pip install inflect -- if you do not have.
    import inflect
    p = inflect.engine()
    
    temp_Str = text.split()
    new_Str = []
    
    for word in temp_Str:
        if word.isdigit():
            temp = p.number_to_words(word)
            new_Str.append(temp)
        else:
            new_Str.append(word)
            
    temp_Str = ' '.join(new_Str)
    return temp_Str


In [15]:
def remTrouble(text):
    new_data = changeNum(text)
    new_data = remPunct(new_data)
    new_data = remWhitespace(new_data)
    new_data = remHTML(new_data)
    new_data = remURL(new_data)
    new_data = remBrackets(new_data)
    new_data = lowerCase(new_data)
    return new_data

In [16]:
## Test string to verify functions work.
exstring = "There are 3 balls in this bag, and 12 in the other one 14 ."

remTrouble(exstring)

'there are three balls in this bag and twelve in the other one fourteen'

In [17]:
## pip install contractions

def contractionReplace(text):
    import contractions
    
    return contractions.fix(text)


## Now its time to tokenize our words to further reduce the dimension of the dataset.
def tokenize(token):
    import nltk
    
    tokens = nltk.word_tokenize(token)
    return tokens

In [18]:
def normalization(text):
    new_data = remTrouble(text)
    new_data = contractionReplace(new_data)
    new_data = tokenize(new_data)
    return new_data

In [19]:
normal_ent = ent.apply(normalization)

In [20]:
## Let's verify our changes!
print(normal_ent[1])

['the', 'enterprise', 'transcripts', 'fight', 'or', 'flight', 'fight', 'or', 'flight', 'mission', 'date', 'may', '6', 'two', 'thousand', 'one', 'hundred', 'and', 'fiftyone', 'original', 'airdate', 'three', 'oct', 'two', 'thousand', 'and', 'one', 'sickbay', 'hoshi', 'is', 'clucking', 'at', 'a', 'slug', 'in', 'a', 'plastic', 'box', 'phlox', 'i', 'did', 'not', 'realise', 'you', 'spoke', 'slug', 'hoshi', 'wish', 'i', 'did', 'she', 'does', 'not', 'look', 'any', 'better', 'does', 'she', 'phlox', 'she', 'we', 'have', 'not', 'been', 'able', 'to', 'determine', 'its', 'gender', 'yet', 'if', 'it', 'has', 'one', 'hoshi', 'i', 'should', 'have', 'left', 'her', 'where', 'i', 'found', 'her', 'phlox', 'nonsense', 'you', 'are', 'on', 'a', 'mission', 'of', 'exploration', 'there', 'is', 'something', 'to', 'be', 'learned', 'from', 'every', 'lifeform', 'hoshi', 'she', 'was', 'not', 'meant', 'to', 'be', 'in', 'this', 'environment', 'phlox', 'i', 'promise', 'to', 'do', 'my', 'best', 'to', 'keep', 'it', 'alive

In [21]:
## Looks great so far! Just need to remove one more thing: STOPWORDS!!!
## We have two methods for that: Lemmatization & Stemming.

### Frequency Distribution 

#### Stopwords:Lemmatization

In [22]:
def stopWord(text):
    import nltk
    from nltk.corpus import stopwords
    nltk.download('stopwords', quiet=True)
    clean = [] ## Create an empty list
    for i in text:
        if i not in stopwords.words('english'):
            clean.append(i)
    return clean


def lemmatize_tokens(text):
    import nltk
    from nltk.stem import WordNetLemmatizer
    nltk.download('wordnet', quiet=True)
    lemmatizer = WordNetLemmatizer()
    lemmas = [] ## Create an empty list to append to.
    for i in text:
        lem = lemmatizer.lemmatize(i,pos='v')
        lemmas.append(lem)
    return lemmas


def preprocess_lemmatize(text):
    stopped = stopWord(text)
    lemmatized = lemmatize_tokens(stopped)
    return lemmatized

In [23]:
lemmatized_lines = normal_ent.apply(preprocess_lemmatize)

In [24]:
print(lemmatized_lines.head(10))

episode 0    [enterprise, transcripts, break, bow, break, b...
episode 1    [enterprise, transcripts, fight, flight, fight...
episode 2    [enterprise, transcripts, strange, new, world,...
episode 3    [enterprise, transcripts, unexpected, unexpect...
episode 4    [enterprise, transcripts, terra, nova, terra, ...
episode 5    [enterprise, transcripts, andorian, incident, ...
episode 6    [enterprise, transcripts, break, ice, break, i...
episode 7    [enterprise, transcripts, civilization, civili...
episode 8    [enterprise, transcripts, fortunate, son, fort...
episode 9    [enterprise, transcripts, cold, front, cold, f...
Name: ENT, dtype: object


#### Stopwords: Stemming

In [25]:

def stem_tokens(text):
    import nltk
    from nltk.stem import LancasterStemmer
    stemmer = LancasterStemmer()
    stems = []
    for i in text:
        stemz = stemmer.stem(i)
        stems.append(stemz)
    return stems


def preprocess_stem(text):
    stopped = stopWord(text)
    stemmed = stem_tokens(stopped)
    return stemmed

In [26]:
stemmed_lines = normal_ent.apply(preprocess_stem)

In [27]:
print(stemmed_lines.head(10))

episode 0    [enterpr, transcrib, brok, bow, brok, bow, mis...
episode 1    [enterpr, transcrib, fight, flight, fight, fli...
episode 2    [enterpr, transcrib, strange, new, world, stra...
episode 3    [enterpr, transcrib, unexpect, unexpect, miss,...
episode 4    [enterpr, transcrib, terr, nov, terr, nov, mis...
episode 5    [enterpr, transcrib, and, incid, and, incid, m...
episode 6    [enterpr, transcrib, break, ic, break, ic, mis...
episode 7    [enterpr, transcrib, civil, civil, miss, dat, ...
episode 8    [enterpr, transcrib, fortun, son, fortun, son,...
episode 9    [enterpr, transcrib, cold, front, cold, front,...
Name: ENT, dtype: object


In [140]:
## Note: Lancaster stemming produces aggressive truncations like "hundr" and "fiftyon".
## This is expected behavior — stemming strips suffixes without checking that the result
## is a real word, unlike lemmatization which returns a dictionary form.

### Frequency Distribution

In [66]:
import nltk

mostPopular = lemmatized_lines.apply(nltk.FreqDist)

In [136]:
## Now, we can iterate through... say the first 10 episodes and find out which words
## were the most popular!

for episode in range(1,10):
    print("Episode #: {}\n----------- \n".format(episode))
    print("\t{}".format(mostPopular[episode].most_common(3)))
    print("\n")

Episode #: 1
----------- 

	[('archer', 155), ('hoshi', 91), ('tpol', 61)]


Episode #: 2
----------- 

	[('tucker', 137), ('archer', 112), ('tpol', 103)]


Episode #: 3
----------- 

	[('tucker', 153), ('archer', 104), ('ahlen', 47)]


Episode #: 4
----------- 

	[('archer', 145), ('tpol', 46), ('jamin', 41)]


Episode #: 5
----------- 

	[('archer', 96), ('tpol', 59), ('tucker', 55)]


Episode #: 6
----------- 

	[('archer', 110), ('tucker', 91), ('tpol', 78)]


Episode #: 7
----------- 

	[('archer', 145), ('tpol', 68), ('riann', 63)]


Episode #: 8
----------- 

	[('ryan', 111), ('archer', 98), ('travis', 69)]


Episode #: 9
----------- 

	[('archer', 147), ('daniels', 70), ('tucker', 62)]




In [138]:
## "Tucker" and "Archer" dominate the first 10 episodes — unsurprising given their
## screen time as the lead engineer and captain on Enterprise.

## Results

- Built a reusable text-normalization pipeline (`normalization()`) composing eight individual functions for punctuation, whitespace, HTML, URL, parenthetical, lowercase, number-to-word, and contraction handling.
- Applied two stopword-aware reduction strategies to the same input:
  - **Lemmatization** (WordNet, verb POS) — preserves dictionary forms, producing human-readable tokens.
  - **Stemming** (Lancaster) — aggressive truncation, producing non-word artifacts (`hundr`, `fiftyon`) but smaller vocabulary.
- Frequency analysis on the lemmatized corpus surfaces character names as the dominant tokens per episode, matching expectations from the show's principal-cast structure.

Lemmatization is the better default for downstream analytics tasks where interpretability matters; stemming would only be preferable for memory-constrained applications where vocabulary collapse outweighs readability.